# RSNA Knee: image-model inference (submission notebook)

Runs the model trained by `src/imaging/train.py` (2.5D ConvNeXt multi-slot MIL) on the hidden test set.
**Internet must be OFF.** Accelerator: **GPU T4** (bf16 is used on Ampere+ GPUs, fp16 otherwise; recent PyTorch builds may not
support the P100). Attach two inputs:
1. the competition data;
2. a private dataset containing the `knee_*.pt` checkpoint(s) and `wheels/` (pylibjpeg, pylibjpeg-libjpeg, pylibjpeg-openjpeg for
   CPython 3.10–3.14, pydicom 3.0.2, timm 1.0.30). Every `knee_*.pt` in that folder is ensembled with equal weight.

Preprocessing is the training preprocessing recorded in each checkpoint (v2 224 px or v6 256 px at 0.6 mm/px; v2 verified bit-identical
to the training cache on 51 local series, v6 is produced by the same function that built the training cache). The run **fails loudly** if decoders are missing or decoding
fails systematically (> 2% of studies). An undecodable series falls back to another series of the same plane or its slot is explicitly
masked; one undecodable slice at either end of a series (≥ 10 slices) is dropped and reported. An image is never silently
zero-filled. Every step is logged to `/kaggle/working/inference_report.json`.

## 1. Environment, discovery and offline dependencies

In [ ]:
import glob, os, subprocess, sys, json, time, shutil
def find_dir(pattern_files, roots=("/kaggle/input",)):
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", pattern_files[0]), recursive=True):
            d = os.path.dirname(p)
            if all(os.path.exists(os.path.join(d, f)) for f in pattern_files):
                return d
    return None
def pip_offline(pkgs):
    whl_dirs = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/*.whl", recursive=True)})
    if not whl_dirs:
        print("no wheel directories attached"); return False
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "-q"] + sum([["--find-links", d] for d in whl_dirs], []) + pkgs
    print("installing", pkgs, "from", whl_dirs); return subprocess.run(cmd).returncode == 0

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'enable a GPU accelerator'

In [ ]:
# Offline dependencies. Install BEFORE pydicom is first imported: pixel-decoder plugins are discovered
# at import time. Wheels in the attached dataset cover CPython 3.10-3.14 (Kaggle image: Python 3.13).
from importlib.metadata import version, PackageNotFoundError
def _ver(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None
print("python", sys.version.split()[0], "| pydicom", _ver("pydicom"), "| pylibjpeg", _ver("pylibjpeg"),
      "| libjpeg", _ver("pylibjpeg-libjpeg"), "| openjpeg", _ver("pylibjpeg-openjpeg"), "| gdcm", _ver("python-gdcm"), "| timm", _ver("timm"))
assert "pydicom" not in sys.modules, "pydicom was imported before the dependency cell: restart the kernel and run from the top"
PYDICOM_VERSION, TIMM_VERSION = "3.0.2", "1.0.30"          # versions used locally (timm: training)
if _ver("pydicom") is None or int(_ver("pydicom").split(".")[0]) < 3:
    pip_offline([f"pydicom=={PYDICOM_VERSION}", "--no-deps"])
missing = [p for p in ("pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg") if _ver(p) is None]
if missing:
    pip_offline(missing)
if _ver("timm") != TIMM_VERSION:
    print(f"timm {_ver('timm')} != {TIMM_VERSION}: installing the bundled wheel")
    pip_offline([f"timm=={TIMM_VERSION}", "--no-deps"])
import pydicom, timm
from pydicom.pixels import get_decoder
from pydicom.uid import JPEGLosslessSV1, JPEGLossless, JPEG2000Lossless, JPEG2000
ok = {u.name: get_decoder(u).is_available for u in (JPEGLosslessSV1, JPEGLossless, JPEG2000Lossless, JPEG2000)}
print("pydicom", pydicom.__version__, "| timm", timm.__version__, "| decoders", ok)
assert all(ok.values()), "JPEG-Lossless / JPEG-2000 decoders unavailable: attach the dataset with the wheels (internet is off)"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} loaded; restart the kernel after installation"

## 2. Configuration

In [ ]:
# ---------------- CONFIG (edit only if auto-discovery fails) ----------------
COMP_DIR = find_dir(["test.csv", "test_series.csv", "sample_submission.csv"]) or "/kaggle/input/rsna-knee-abnormality-detection"
MODEL_DIR = None        # folder with the trained *.pt checkpoints (auto: any attached folder containing knee_*.pt)
TTA_SHIFT = True        # adopted: +0.0012 (E1) and +0.0017 [+0.0001, +0.0033] (E1p) pseudo-val; see experiments report
if MODEL_DIR is None:
    cands = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/knee_*.pt", recursive=True)})
    MODEL_DIR = cands[0] if cands else None
CFG = dict(test_csv=f"{COMP_DIR}/test.csv", test_series_csv=f"{COMP_DIR}/test_series.csv",
           test_dicom_root=f"{COMP_DIR}/test_series", sample_submission=f"{COMP_DIR}/sample_submission.csv",
           model_dir=MODEL_DIR, cache_dir="/tmp/knee_cache", out_path="/kaggle/working/submission.csv",
           workers=max(1, (os.cpu_count() or 2)), batch_size=4, device="cuda",
           tta_shift=TTA_SHIFT)     # average a 2nd window bag shifted by half a window step (see the experiments report)
print(json.dumps(CFG, indent=1))
assert MODEL_DIR is not None, "attach the dataset that contains the knee_*.pt checkpoints"

## 3. Code (generated from `src/imaging/*.py`; identical to the locally tested modules)

In [ ]:
import os, sys, math, time, json
import numpy as np, pandas as pd, torch, torch.nn as nn
"""Build the per-study index used by the image models (train) and the same logic for test.

For every study it decides:
  * which series fills each slot  (plane x {fluid-sensitive, non-fluid-sensitive}),
  * the knee side used for orientation normalisation (+ provenance and conflict flags),
  * the 2.5D neighbour offset per series from its physical slice spacing,
  * the CV fold (train studies only).

Series *metadata* (plane, Fluid_Sensitive flag, slice count, spacing, geometry) is used only
to choose WHICH images are read. It is never fed to the classifier as a feature.

Laterality policy (per study):
  side = majority DICOM Laterality tag if any series carries one, else
         majority geometric side over series whose volume centre is >= 20 mm from x=0
         (sign of patient-x: + -> left knee), else majority over all series.
  Flags: 'conflict' if a confident geometric side disagrees with the tag side;
         'ambiguous' if no series is >= 20 mm off the midline.
"""
import argparse
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

PLANES = ["Sagittal", "Coronal", "Axial"]
SLOTS = [(p, fs) for fs in (1, 0) for p in PLANES]          # 0..2 = FS sag/cor/ax, 3..5 = non-FS
SLOT_NAMES = [f"{p[:3].lower()}_{'fs' if fs else 'nfs'}" for p, fs in SLOTS]
TARGET_STEP_MM = 3.5      # physical distance between the 3 slices of a 2.5D window
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]


def choose_slots(ser):
    """ser: DataFrame of one study's series with columns SeriesInstanceUID, Anatomical_Plane,
    Fluid_Sensitive, n_slices. Returns dict slot_idx -> SeriesInstanceUID (or None) and the
    candidate lists (for optional random choice during training)."""
    out, cands = {}, {}
    for si, (plane, fs) in enumerate(SLOTS):
        c = ser[(ser.Anatomical_Plane == plane) & (ser.Fluid_Sensitive == fs) & (ser.n_slices >= 3)]
        c = c.sort_values(["n_slices", "SeriesInstanceUID"], ascending=[False, True])
        cands[si] = list(c.SeriesInstanceUID)
        out[si] = cands[si][0] if cands[si] else None
    return out, cands


def study_side(lat_rows):
    """lat_rows: rows of laterality_by_series for one study."""
    tags = lat_rows.tag_side.dropna()
    tags = tags[tags.isin(["L", "R"])]
    # Geometry only from coronal/axial series: their in-plane centre-x is exact for every slice,
    # whereas sagittal slices are spread along x (v2 index fix; see image_pipeline_audit.md).
    ca = lat_rows[lat_rows.plane.isin(["Coronal", "Axial"])]
    ca = ca if len(ca) else lat_rows
    conf = ca[ca.center_x_mm.abs() >= 20]
    geo_src = conf if len(conf) else ca
    geo = geo_src.geo_side.value_counts().idxmax()
    ambiguous = len(conf) == 0
    if len(tags):
        tag = tags.value_counts().idxmax()
        conflict = (not ambiguous) and (conf.geo_side.value_counts().idxmax() != tag)
        return tag, "tag", bool(conflict), ambiguous
    return geo, "geometry", False, ambiguous


def build(series_df, headers_df, lat_df):
    s = series_df.merge(headers_df, on="SeriesInstanceUID", how="left")
    rows = []
    for uid, ser in s.groupby("StudyInstanceUID"):
        slots, cands = choose_slots(ser)
        side, src, conflict, amb = study_side(lat_df[lat_df.StudyInstanceUID == uid])
        r = {"StudyInstanceUID": uid, "side": side, "side_source": src, "side_conflict": conflict,
             "side_ambiguous": amb}
        for si, name in enumerate(SLOT_NAMES):
            sid = slots[si]
            r[f"{name}_uid"] = sid
            r[f"{name}_n_cands"] = len(cands[si])
            r[f"{name}_cands"] = "|".join(cands[si])
            if sid is not None:
                h = ser[ser.SeriesInstanceUID == sid].iloc[0]
                sp = h.spacing_mm if np.isfinite(h.spacing_mm) and h.spacing_mm > 0 else 3.5
                r[f"{name}_n"] = int(h.n_slices)
                r[f"{name}_spacing"] = float(sp)
                r[f"{name}_step"] = int(max(1, round(TARGET_STEP_MM / sp)))
            else:
                r[f"{name}_n"], r[f"{name}_spacing"], r[f"{name}_step"] = 0, np.nan, 0
        rows.append(r)
    return pd.DataFrame(rows)


def add_folds(idx, labels, lang, dup_groups, n_folds=5, seed=42):
    """Stratify non-gold studies by language x number of report-positive findings.
    Gold studies get fold -1 (never in a training/validation fold). Duplicate studies share a fold."""
    lab = labels.set_index("StudyInstanceUID")
    gold = set(lab.index[lab.label_source == "gold"])
    ng = idx[~idx.StudyInstanceUID.isin(gold)].copy()
    burden = (lab.loc[ng.StudyInstanceUID, TARGETS] >= 0.5).sum(axis=1).clip(upper=6).astype(str).values
    key = (lang.reindex(ng.StudyInstanceUID).fillna("NA").values + "_" + burden)
    vc = pd.Series(key).value_counts()
    key = np.where(pd.Series(key).map(vc).values < n_folds, "rare", key)
    folds = np.full(len(ng), -1)
    for f, (_, va) in enumerate(StratifiedKFold(n_folds, shuffle=True, random_state=seed).split(ng, key)):
        folds[va] = f
    ng["fold"] = folds
    fmap = dict(zip(ng.StudyInstanceUID, ng.fold))
    for grp in dup_groups:                       # force duplicates into the first member's fold
        f0 = fmap.get(grp[0])
        for u in grp[1:]:
            if u in fmap and f0 is not None:
                fmap[u] = f0
    idx["fold"] = idx.StudyInstanceUID.map(fmap).fillna(-1).astype(int)
    idx["is_gold"] = idx.StudyInstanceUID.isin(gold)
    return idx


def headers_table(series_headers_csv):
    h = pd.read_csv(series_headers_csv, dtype=str)
    out = pd.DataFrame({"SeriesInstanceUID": h.SeriesInstanceUID_dir,
                        "n_slices": pd.to_numeric(h.n_files),
                        "spacing_mm": pd.to_numeric(h.SpacingBetweenSlices, errors="coerce").fillna(
                            pd.to_numeric(h.SliceThickness, errors="coerce"))})
    return out

In [ ]:
"""DICOM -> model-input preprocessing for inference (and for re-building caches).

Reproduces the v2 training cache exactly (verified bit-exact on the local test studies by
src/imaging/check_infer_exactness.py):
  order slices by ImagePositionPatient . slice-normal (fallback InstanceNumber, then name)
  -> RescaleSlope/Intercept, MONOCHROME1 inversion, middle frame of multi-frame files
  -> per-series 1st/99th percentile window -> uint8 -> PIL bilinear resize to 224x224.
Differences from v2, on purpose: v2 silently skipped ANY slice that failed to decode. Here only one
undecodable slice at either end of the slice order is dropped, and it is reported; any other failure
raises DecodeError (see decode_series). An unreadable series is reported, never replaced by zeros.

The study index for unseen studies (slots, knee side, 2.5D step) is rebuilt from the DICOM
headers with the same rules as src/imaging/study_index.py.
"""
import os

import numpy as np
import pandas as pd
import pydicom
from PIL import Image




class DecodeError(RuntimeError):
    pass


def process_pool(workers):
    """Process pool that also works for functions defined inside a notebook: 'fork' where available
    (Linux/Kaggle; Python 3.14 changed the POSIX default to 'forkserver', which cannot pickle them)."""
    import multiprocessing as mp
    from concurrent.futures import ProcessPoolExecutor
    if "fork" in mp.get_all_start_methods():
        return ProcessPoolExecutor(workers, mp_context=mp.get_context("fork"))
    return ProcessPoolExecutor(workers)


def order_series_files(series_dir):
    fnames = [f for f in os.listdir(series_dir) if f.lower().endswith(".dcm")]
    if not fnames:
        raise DecodeError(f"no .dcm files in {series_dir}")
    entries = []
    for f in fnames:
        try:
            ds = pydicom.dcmread(os.path.join(series_dir, f))
        except Exception as ex:
            raise DecodeError(f"unreadable header {series_dir}/{f}: {type(ex).__name__}: {ex}")
        entries.append([None, f, ds])
    have_geo = True
    for e in entries:
        try:
            iop = np.array(e[2].ImageOrientationPatient, dtype=np.float64)
            ipp = np.array(e[2].ImagePositionPatient, dtype=np.float64)
            e[0] = float(np.dot(np.cross(iop[:3], iop[3:]), ipp))
        except Exception:
            have_geo = False
            break
    if have_geo and len({round(e[0], 3) for e in entries}) > 1:
        entries.sort(key=lambda e: e[0])
        return [(e[1], e[2]) for e in entries]
    try:
        nums = [int(getattr(e[2], "InstanceNumber", -1)) for e in entries]
        if len(set(nums)) == len(entries) and all(n >= 0 for n in nums):
            order = sorted(range(len(entries)), key=lambda i: nums[i])
            return [(entries[i][1], entries[i][2]) for i in order]
    except Exception:
        pass
    entries.sort(key=lambda e: e[1])
    return [(e[1], e[2]) for e in entries]


def decode_slice(ds):
    arr = ds.pixel_array.astype(np.float32)        # raises if no decoder is available
    if arr.ndim == 3:
        arr = arr[arr.shape[0] // 2]
    arr = arr * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def decode_series(ordered, series_dir):
    """Decode every slice of an ordered series -> (list of float arrays, list of dropped file names).

    Exactly ONE undecodable slice at either END of the slice order, in a series of >= 10 slices, is
    dropped and reported (seen twice in the 24k training series: a truncated first/last file; the
    legacy v2 cache dropped the same two slices). Any other decode failure raises DecodeError, so the
    caller falls back to another series or masks the slot; nothing is ever zero-filled."""
    slices, errs = [], []
    for i, (f, ds) in enumerate(ordered):
        try:
            slices.append(decode_slice(ds))
        except Exception as ex:
            errs.append((i, f, ds, ex))
    if not errs:
        return slices, []
    i, f, ds, ex = errs[0]
    if len(errs) == 1 and i in (0, len(ordered) - 1) and len(ordered) >= 10:
        return slices, [f]
    raise DecodeError(f"pixel decode failed {series_dir}/{f} (transfer syntax {getattr(ds.file_meta, 'TransferSyntaxUID', '?')}): "
                      f"{type(ex).__name__}: {ex} [{len(errs)} of {len(ordered)} slices failed]")


def preprocess_series(series_dir, img_size=224):
    """-> (uint8 [T, img_size, img_size], geometry dict). Raises DecodeError on failure (see decode_series)."""
    ordered = order_series_files(series_dir)
    slices, dropped = decode_series(ordered, series_dir)
    shapes = {s.shape for s in slices}
    if len(shapes) != 1:
        raise DecodeError(f"inconsistent slice shapes {shapes} in {series_dir}")
    stack = np.stack(slices, 0)
    lo, hi = np.percentile(stack, [1.0, 99.0])
    hi = max(hi, lo + 1.0)
    stack = (np.clip((stack - lo) / (hi - lo), 0, 1) * 255.0).astype(np.uint8)
    out = np.stack([np.array(Image.fromarray(s).resize((img_size, img_size), Image.BILINEAR), dtype=np.uint8)
                    for s in stack])
    if not np.isfinite(out).all() or out.max() == out.min():
        raise DecodeError(f"degenerate (constant) series {series_dir}")
    return out, {**series_geometry(ordered), "dropped_slices": dropped}


def preprocess_series_v6(series_dir, out_size=256, target_mm=0.6):
    """v6: same decode/order/percentile as v2, then ASPECT-PRESERVING physical resampling to
    `target_mm` per pixel (PIL bilinear) and a centred out_size x out_size crop/zero-pad
    (256 x 0.6 mm = 153.6 mm field of view). -> (uint8 [T, out, out], geometry dict)."""
    ordered = order_series_files(series_dir)
    slices, dropped = decode_series(ordered, series_dir)
    shapes = {s.shape for s in slices}
    if len(shapes) != 1:
        raise DecodeError(f"inconsistent slice shapes {shapes} in {series_dir}")
    stack = np.stack(slices, 0)
    lo, hi = np.percentile(stack, [1.0, 99.0])
    hi = max(hi, lo + 1.0)
    stack = (np.clip((stack - lo) / (hi - lo), 0, 1) * 255.0).astype(np.uint8)
    ds0 = ordered[0][1]
    try:
        ps_r, ps_c = [float(v) for v in ds0.PixelSpacing]
    except Exception:
        ps_r = ps_c = 160.0 / max(stack.shape[1:])            # assume a 160 mm FOV if spacing is missing
    H, W = stack.shape[1:]
    nh, nw = max(1, int(round(H * ps_r / target_mm))), max(1, int(round(W * ps_c / target_mm)))
    out = np.zeros((len(stack), out_size, out_size), np.uint8)
    h, w = min(out_size, nh), min(out_size, nw)                 # copied extent
    sy, sx = max(0, (nh - out_size) // 2), max(0, (nw - out_size) // 2)   # centred crop in the resized image
    dy, dx = max(0, (out_size - nh) // 2), max(0, (out_size - nw) // 2)   # centred placement (zero pad)
    for i, s in enumerate(stack):
        r = np.array(Image.fromarray(s).resize((nw, nh), Image.BILINEAR), dtype=np.uint8)
        out[i, dy:dy + h, dx:dx + w] = r[sy:sy + h, sx:sx + w]
    if out.max() == out.min():
        raise DecodeError(f"degenerate (constant) series {series_dir}")
    return out, {**series_geometry(ordered), "dropped_slices": dropped}


def series_geometry(ordered):
    ds0 = ordered[0][1]
    geo = {"n_slices": len(ordered), "tag_side": "", "center_x_mm": np.nan, "spacing_mm": np.nan}
    lat = str(getattr(ds0, "Laterality", "") or "").strip().upper()[:1]
    geo["tag_side"] = lat if lat in ("L", "R") else ""
    try:
        iop = np.array(ds0.ImageOrientationPatient, dtype=float)
        ps = [float(v) for v in ds0.PixelSpacing]
        ipp = np.array(ds0.ImagePositionPatient, dtype=float)
        c = ipp + iop[:3] * ps[1] * int(ds0.Columns) / 2 + iop[3:] * ps[0] * int(ds0.Rows) / 2
        geo["center_x_mm"] = float(c[0])
    except Exception:
        pass
    # Same source priority as the training index (study_index.headers_table):
    # SpacingBetweenSlices -> SliceThickness -> measured position spacing -> 3.5 mm.
    for tag in ("SpacingBetweenSlices", "SliceThickness"):
        try:
            v = float(getattr(ds0, tag))
            if np.isfinite(v) and v > 0:
                geo["spacing_mm"] = v
                break
        except Exception:
            continue
    if not np.isfinite(geo["spacing_mm"]):
        try:
            iop = np.array(ds0.ImageOrientationPatient, dtype=float)
            n = np.cross(iop[:3], iop[3:])
            pos = [float(np.dot(n, np.array(d.ImagePositionPatient, dtype=float))) for _, d in ordered]
            geo["spacing_mm"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 3.5
        except Exception:
            geo["spacing_mm"] = 3.5
    return geo


def series_header_geometry(series_dir):
    """Header-only pass (no pixel decoding): slice count, tag side, centre-x, spacing."""
    fnames = [f for f in os.listdir(series_dir) if f.lower().endswith(".dcm")]
    if not fnames:
        raise DecodeError(f"no .dcm files in {series_dir}")
    entries = []
    for f in fnames:
        try:
            ds = pydicom.dcmread(os.path.join(series_dir, f), stop_before_pixels=True)
        except Exception as ex:
            raise DecodeError(f"unreadable header {series_dir}/{f}: {type(ex).__name__}: {ex}")
        try:
            iop = np.array(ds.ImageOrientationPatient, dtype=float)
            key = float(np.dot(np.cross(iop[:3], iop[3:]), np.array(ds.ImagePositionPatient, dtype=float)))
        except Exception:
            key = 0.0
        entries.append((key, f, ds))
    entries.sort(key=lambda e: e[0])
    return series_geometry([(f, ds) for _, f, ds in entries])


def study_side_from_geo(geos):
    """geos: list of geometry dicts (with 'plane') for one study's series; same policy as
    study_index.study_side: tag majority if any, else coronal/axial geometric majority."""
    tags = [g["tag_side"] for g in geos if g["tag_side"] in ("L", "R")]
    ca = [g for g in geos if g.get("plane") in ("Coronal", "Axial") and np.isfinite(g["center_x_mm"])]
    ca = ca if ca else [g for g in geos if np.isfinite(g["center_x_mm"])]
    conf = [g for g in ca if abs(g["center_x_mm"]) >= 20]
    src = conf if conf else ca
    geo_votes = ["L" if g["center_x_mm"] > 0 else "R" for g in src]
    geo = max(set(geo_votes), key=geo_votes.count) if geo_votes else "L"
    if tags:
        tag = max(set(tags), key=tags.count)
        conflict = bool(conf) and (max(set(["L" if g["center_x_mm"] > 0 else "R" for g in conf]),
                                       key=["L" if g["center_x_mm"] > 0 else "R" for g in conf].count) != tag)
        return tag, "tag", conflict, not conf
    return geo, "geometry", False, not conf


def build_index_row(study_uid, series_meta, geos):
    """series_meta: DataFrame (SeriesInstanceUID, Anatomical_Plane, Fluid_Sensitive) for one study;
    geos: dict series_uid -> geometry dict (from preprocess_series)."""
    ser = series_meta.copy()
    ser["n_slices"] = ser.SeriesInstanceUID.map(lambda s: geos[s]["n_slices"] if s in geos else 0)
    slots, cands = choose_slots(ser)
    plane_of = dict(zip(ser.SeriesInstanceUID, ser.Anatomical_Plane))
    side, src, conflict, amb = study_side_from_geo([{**geos[s], "plane": plane_of[s]} for s in ser.SeriesInstanceUID if s in geos])
    r = {"StudyInstanceUID": study_uid, "side": side, "side_source": src, "side_conflict": conflict, "side_ambiguous": amb}
    for si, name in enumerate(SLOT_NAMES):
        sid = slots[si]
        r[f"{name}_uid"], r[f"{name}_n_cands"], r[f"{name}_cands"] = sid, len(cands[si]), "|".join(cands[si])
        if sid is not None:
            sp = geos[sid]["spacing_mm"]
            r[f"{name}_n"], r[f"{name}_spacing"] = geos[sid]["n_slices"], sp
            r[f"{name}_step"] = int(max(1, round(TARGET_STEP_MM / sp)))
        else:
            r[f"{name}_n"], r[f"{name}_spacing"], r[f"{name}_step"] = 0, np.nan, 0
    return r

In [ ]:
"""Study-level dataset for 2.5D multi-slot MRI models.

Reads per-series uint8 volumes (T, H, W) from a preprocessing cache (default: the verified
v2 cache: per-series 1-99 percentile window -> uint8 -> 224x224 bilinear, slices sorted
along the slice normal). Orientation normalisation (applied here, at load time):

  * coronal / axial, right knee  -> mirror columns  (lateral is then always image-right)
  * sagittal, right knee         -> reverse slice order (slice 0 is then always lateral)

The sagittal rule follows from the data's uniform orientation (rows +y, cols -z, slice
normal -x for every sagittal series): ascending normal projection runs from patient-left
to patient-right, i.e. lateral->medial for a left knee and medial->lateral for a right knee.

Each slot yields K windows of 3 slices [c-step, c, c+step] (step from physical spacing, about
3.5 mm). Missing slots are zeros with mask 0; the model must ignore them.
"""
import os

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset




def window_centres(T, K, lo, hi, train, rng):
    a, b = lo * (T - 1), hi * (T - 1)
    if K == 1:
        c = np.array([(a + b) / 2])
    elif train:
        edges = np.linspace(a, b, K + 1)
        c = rng.uniform(edges[:-1], edges[1:])
    else:
        c = np.linspace(a, b, K)
    return np.clip(np.round(c).astype(int), 0, T - 1)


class KneeStudyDataset(Dataset):
    def __init__(self, index_df, cache_dir, slots=(0, 1, 2), K=8, train=False, labels=None,
                 weights=None, lo=0.04, hi=0.96, random_series=False, spacing_map=None, seed=0,
                 orient=True, strict=True, fallback=True, sag_pos_fix=False):
        self.idx = index_df.reset_index(drop=True)
        self.cache_dir = cache_dir
        self.slots = list(slots)
        self.K, self.train, self.lo, self.hi = K, train, lo, hi
        self.labels = labels            # (N, 12) float array aligned with index_df, or None
        self.weights = weights          # (N,) float array or None
        self.random_series = random_series
        self.spacing_map = spacing_map or {}
        self.seed = seed
        self.orient = orient
        self.strict = strict            # missing cache file for a chosen series -> raise, never zero-fill
        self.fallback = fallback        # empty FS slot (0-2) -> use the same plane's non-FS series (slot+3)
        # Slice-position feature for RIGHT-knee sagittal series. The mirrored window indices already put
        # window k at anatomical position p = c/(T-1) (0 = lateral) for both sides, so the correct feature
        # is p. The original code passed 1 - p (found 2026-10-17 by attention_map.py); False reproduces it
        # for checkpoints trained that way (args without "sag_pos_fix").
        self.sag_pos_fix = sag_pos_fix

    def __len__(self):
        return len(self.idx)

    def _load(self, uid):
        p = os.path.join(self.cache_dir, f"{uid}.npy")
        if not os.path.exists(p):
            if self.strict:
                raise FileNotFoundError(f"cache missing for series {uid}")
            return None
        return np.load(p, mmap_mode="r")

    def __getitem__(self, i):
        row = self.idx.iloc[i]
        rng = np.random.default_rng((self.seed * 1_000_003 + i * 7919 + int(torch.randint(0, 1 << 30, (1,)))) if self.train else i)
        S = len(self.slots)
        x, mask, pos = None, np.zeros(S, np.float32), np.zeros((S, self.K), np.float32)
        right = (row["side"] == "R") and self.orient
        for j, si in enumerate(self.slots):
            name = SLOT_NAMES[si]
            uid = row[f"{name}_uid"]
            if not isinstance(uid, str) and self.fallback and si < 3 and (si + 3) not in self.slots:
                name = SLOT_NAMES[si + 3]          # same plane, non-fluid-sensitive series
                uid = row[f"{name}_uid"]
            if not isinstance(uid, str):
                continue
            if self.train and self.random_series:
                cands = str(row[f"{name}_cands"]).split("|")
                uid = cands[rng.integers(len(cands))]
            step = int(row[f"{name}_step"]) if uid == row[f"{name}_uid"] else \
                max(1, int(round(3.5 / self.spacing_map.get(uid, 3.5))))
            arr = self._load(uid)
            if arr is None:
                continue
            T, H, W = arr.shape
            if x is None:
                x = np.zeros((S, self.K, 3, H, W), np.uint8)
            plane = SLOTS[si][0]
            cs = window_centres(T, self.K, self.lo, self.hi, self.train, rng)
            for k, c in enumerate(cs):
                ids = np.clip(np.array([c - step, c, c + step]), 0, T - 1)
                if right and plane == "Sagittal":
                    ids = (T - 1) - ids[::-1]          # reversed slice order, channel order lateral->medial
                win = np.asarray(arr[ids])
                if right and plane in ("Coronal", "Axial"):
                    win = win[:, :, ::-1]
                x[j, k] = win
                p = c / max(T - 1, 1)
                pos[j, k] = (1 - p) if (right and plane == "Sagittal" and not self.sag_pos_fix) else p
            mask[j] = 1.0
        if x is None:
            raise RuntimeError(f"study {row['StudyInstanceUID']} has no usable series")
        out = {"x": torch.from_numpy(x), "mask": torch.from_numpy(mask), "pos": torch.from_numpy(pos), "i": i}
        if self.labels is not None:
            out["y"] = torch.from_numpy(self.labels[i].astype(np.float32))
        if self.weights is not None:
            out["w"] = torch.tensor(float(self.weights[i]), dtype=torch.float32)
        return out


def gpu_augment(x, mask, rot_deg=10.0, scale=0.10, shift=0.06, gamma=0.20, gain=0.10):
    """x: float [B,S,K,3,H,W] in [0,1]. One random affine + gamma/gain per (study, slot),
    shared by all windows of that series (keeps the 3 slices of a window aligned)."""
    B, S, K, C, H, W = x.shape
    n = B * S
    dev = x.device
    ang = (torch.rand(n, device=dev) * 2 - 1) * np.deg2rad(rot_deg)
    sc = 1 + (torch.rand(n, device=dev) * 2 - 1) * scale
    tx, ty = [(torch.rand(n, device=dev) * 2 - 1) * shift for _ in range(2)]
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    xs = x.reshape(n, K * C, H, W)
    grid = torch.nn.functional.affine_grid(theta, xs.shape, align_corners=False)
    xs = torch.nn.functional.grid_sample(xs, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    g = torch.exp((torch.rand(n, 1, 1, 1, device=dev) * 2 - 1) * gamma)
    a = 1 + (torch.rand(n, 1, 1, 1, device=dev) * 2 - 1) * gain
    xs = (xs.clamp(0, 1) ** g * a).clamp(0, 1)
    return xs.reshape(B, S, K, C, H, W)


def load_labels(label_csv, uids, transform="soft"):
    lab = pd.read_csv(label_csv).set_index("StudyInstanceUID").reindex(uids)
    y = lab[TARGETS].values.astype(np.float32)
    if np.isnan(y).any():
        raise ValueError(f"{int(np.isnan(y).sum())} missing label cells for the requested studies")
    if transform == "hard":
        y = (y >= 0.5).astype(np.float32)
    return y, lab

In [ ]:
"""2.5D multi-slot MIL model: shared pretrained 2D encoder on 3-slice windows, token
embeddings (slot + slice position), and a choice of study-level aggregation.

Aggregators (agg=):
  mean   masked mean over all window tokens         -> linear 12
  max    masked max over all window tokens          -> linear 12
  gated  Ilse et al. 2018 gated attention, 1 query  -> linear 12
  attn12 per-finding attention (12 queries), each finding scored from its own pooled vector
  tfm    2-layer transformer over tokens, then attn12
"""
import math

import timm
import torch
import torch.nn as nn

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class KneeMIL(nn.Module):
    def __init__(self, backbone="convnext_tiny.fb_in22k_ft_in1k", pretrained=True, n_slots=3, n_out=12,
                 dim=384, agg="attn12", use_pos=True, use_slot=True, drop_path=0.1, grad_ckpt=False):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, drop_path_rate=drop_path)
        if grad_ckpt:
            self.enc.set_grad_checkpointing(True)
        nf = self.enc.num_features
        self.proj = nn.Sequential(nn.Dropout(0.2), nn.Linear(nf, dim), nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(n_slots, dim) if use_slot else None
        self.pos_mlp = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim)) if use_pos else None
        self.agg, self.n_out = agg, n_out
        if agg in ("mean", "max"):
            self.head = nn.Linear(dim, n_out)
        elif agg == "gated":
            self.att_v = nn.Linear(dim, 128)
            self.att_u = nn.Linear(dim, 128)
            self.att_w = nn.Linear(128, 1)
            self.head = nn.Linear(dim, n_out)
        elif agg in ("attn12", "tfm"):
            if agg == "tfm":
                layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
                self.tfm = nn.TransformerEncoder(layer, 2)
            self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(0.1), nn.Linear(256, n_out))
            self.cls_w = nn.Parameter(torch.randn(n_out, dim) / math.sqrt(dim))
            self.cls_b = nn.Parameter(torch.zeros(n_out))
        else:
            raise ValueError(agg)
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1), persistent=False)

    def encode(self, x01):
        """x01: float [N, 3, H, W] in [0,1] -> [N, nf]"""
        return self.enc((x01 - self.mean) / self.std)

    def forward(self, x01, mask, pos):
        """x01: float [B,S,K,3,H,W] in [0,1]; mask [B,S]; pos [B,S,K] -> logits [B,12]"""
        B, S, K = x01.shape[:3]
        valid = mask > 0
        feats = x01.new_zeros(B, S, K, self.proj[1].out_features)
        xv = x01[valid]                                          # [Nv, K, 3, H, W]
        f = self.encode(xv.flatten(0, 1).contiguous(memory_format=torch.channels_last))
        feats[valid] = self.proj(f.float()).view(xv.shape[0], K, -1).to(feats.dtype)
        tok = feats
        if self.slot_emb is not None:
            tok = tok + self.slot_emb.weight[:S].view(1, S, 1, -1)
        if self.pos_mlp is not None:
            tok = tok + self.pos_mlp(pos.unsqueeze(-1))
        tok = tok.flatten(1, 2)                                  # [B, S*K, D]
        tmask = valid.unsqueeze(-1).expand(B, S, K).flatten(1)   # True = real token
        if self.agg == "mean":
            m = tmask.unsqueeze(-1).float()
            return self.head((tok * m).sum(1) / m.sum(1).clamp(min=1))
        if self.agg == "max":
            return self.head(tok.masked_fill(~tmask.unsqueeze(-1), -1e4).max(1).values)
        if self.agg == "gated":
            a = self.att_w(torch.tanh(self.att_v(tok)) * torch.sigmoid(self.att_u(tok))).squeeze(-1)
            a = torch.softmax(a.masked_fill(~tmask, -1e4), 1)
            return self.head((a.unsqueeze(-1) * tok).sum(1))
        if self.agg == "tfm":
            tok = self.tfm(tok, src_key_padding_mask=~tmask)
        a = self.att(tok).masked_fill(~tmask.unsqueeze(-1), -1e4)  # [B, T, 12]
        a = torch.softmax(a, 1)
        pooled = torch.einsum("btc,btd->bcd", a, tok)               # [B, 12, D]
        return (pooled * self.cls_w).sum(-1) + self.cls_b

In [ ]:
"""Hidden-test inference: DICOM -> study index -> selected-series cache -> model ensemble -> submission.

Failure policy (never zero-fill):
  * missing pixel decoders                      -> abort (assert_decoders)
  * a series that cannot be decoded             -> try the next candidate series of the same slot;
                                                   if none is left the slot is explicitly MASKED
                                                   (the model was trained with masks) and logged
  * a study with no decodable series at all     -> abort
  * more than MAX_FAILED_STUDY_FRACTION of studies with any failure -> abort (systematic problem)

Each checkpoint records the preprocessing it was trained with (args["preproc"]: "v2" = 224 px squash,
"v6" = 0.6 mm/px aspect-preserving 256 px crop; older checkpoints without the key are v2). Series are
decoded once per preprocessing version into cache_dir/<version>/, so v2 and v6 models can be ensembled.
"""
import json
import os
import time

import numpy as np
import pandas as pd
import torch






MAX_FAILED_STUDY_FRACTION = 0.02


def assert_decoders():
    from pydicom.pixels import get_decoder
    from pydicom.uid import JPEG2000, JPEG2000Lossless, JPEGLossless, JPEGLosslessSV1
    missing = [u.name for u in (JPEGLosslessSV1, JPEGLossless, JPEG2000Lossless, JPEG2000) if not get_decoder(u).is_available]
    if missing:
        raise RuntimeError(f"No pixel decoder for {missing}: install pylibjpeg, pylibjpeg-libjpeg, pylibjpeg-openjpeg (or gdcm)")


def _geo_job(args):
    sdir, sid = args
    try:
        return sid, series_header_geometry(sdir), None
    except Exception as ex:
        return sid, None, f"{type(ex).__name__}: {ex}"


def build_test_index(series_csv, dicom_root, workers=4, log=print):
    ser = pd.read_csv(series_csv)
    jobs = [(os.path.join(dicom_root, r.StudyInstanceUID, r.SeriesInstanceUID), r.SeriesInstanceUID) for r in ser.itertuples()]
    geos, failures = {}, []
    with process_pool(workers) as ex:
        for sid, g, err in ex.map(_geo_job, jobs, chunksize=8):
            if g is None:
                failures.append({"SeriesInstanceUID": sid, "stage": "header", "error": err})
            else:
                geos[sid] = g
    rows = []
    for st, meta in ser.groupby("StudyInstanceUID"):
        ok = meta[meta.SeriesInstanceUID.isin(geos)]
        if ok.empty:
            raise RuntimeError(f"study {st}: no readable series headers")
        rows.append(build_index_row(st, ok, geos))
    log(f"test index: {len(rows)} studies, {len(geos)} series headers ok, {len(failures)} header failures")
    return pd.DataFrame(rows), failures, geos


def _decode_job(args):
    """-> (series uid, error or None, dropped end-slice file names)"""
    sdir, sid, out_path, preproc = args
    if os.path.exists(out_path):
        return sid, None, []
    try:
        arr, geo = preprocess_series_v6(sdir) if preproc == "v6" else preprocess_series(sdir, 224)
        tmp = out_path + ".tmp.npy"            # atomic: an interrupted write never looks like a cached series
        np.save(tmp, arr)
        os.replace(tmp, out_path)
        return sid, None, geo.get("dropped_slices", [])
    except Exception as ex:
        return sid, f"{type(ex).__name__}: {ex}", []


def preprocess_selected(index_df, geos, dicom_root, cache_dir, slots, workers=4, preproc="v2", log=print):
    """Decode only the series needed for `slots`. Candidate order per slot: the slot's own
    candidates, then (FS slots whose non-FS twin is not itself a model slot) the plane's non-FS
    candidates. On a decode failure the next candidate is tried; if none is left the slot is
    explicitly masked. UID and 2.5D-step columns are rewritten to what was actually decoded."""
    os.makedirs(cache_dir, exist_ok=True)
    idx = index_df.copy().reset_index(drop=True)
    plan = {}
    for ri, r in idx.iterrows():
        for si in slots:
            own = [c for c in str(r[f"{SLOT_NAMES[si]}_cands"] or "").split("|") if c and c != "nan"]
            fb = []
            if si < 3 and (si + 3) not in slots:
                fb = [c for c in str(r[f"{SLOT_NAMES[si + 3]}_cands"] or "").split("|") if c and c != "nan"]
            plan[(ri, si)] = [(si, c) for c in own] + [(si + 3, c) for c in fb]
    chosen = {k: None for k in plan}
    failures, failed_studies = [], set()
    with process_pool(workers) as ex:
        while True:
            jobs = {k: v[0] for k, v in plan.items() if chosen[k] is None and v}
            if not jobs:
                break
            args = [(os.path.join(dicom_root, idx.at[k[0], "StudyInstanceUID"], c), c,
                     os.path.join(cache_dir, f"{c}.npy"), preproc) for k, (_, c) in jobs.items()]
            for k, (sid, err, dropped) in zip(list(jobs.keys()), ex.map(_decode_job, args, chunksize=4)):
                if dropped:                     # non-fatal: one undecodable end slice dropped (infer_lib.decode_series)
                    failures.append({"StudyInstanceUID": idx.at[k[0], "StudyInstanceUID"], "slot": SLOT_NAMES[k[1]],
                                     "SeriesInstanceUID": sid, "stage": "dropped_end_slice", "error": "|".join(dropped)})
                if err is None:
                    chosen[k] = jobs[k]
                else:
                    failures.append({"StudyInstanceUID": idx.at[k[0], "StudyInstanceUID"], "slot": SLOT_NAMES[k[1]],
                                     "SeriesInstanceUID": sid, "stage": "decode", "error": err})
                    failed_studies.add(idx.at[k[0], "StudyInstanceUID"])
                    plan[k] = plan[k][1:]
    for (ri, si), ch in chosen.items():
        for col in {si, si + 3} if si < 3 and (si + 3) not in slots else {si}:
            idx.at[ri, f"{SLOT_NAMES[col]}_uid"] = None
        if ch is None:
            continue
        col, sid = ch
        idx.at[ri, f"{SLOT_NAMES[col]}_uid"] = sid
        idx.at[ri, f"{SLOT_NAMES[col]}_step"] = int(max(1, round(3.5 / geos[sid]["spacing_mm"])))
    n = len(idx)
    for ri, k_ok in pd.Series({ri: any(chosen[(ri, si)] is not None for si in slots) for ri in range(n)}).items():
        if not k_ok:
            raise RuntimeError(f"study {idx.at[ri, 'StudyInstanceUID']}: no decodable series for any model slot")
    if len(failed_studies) / max(n, 1) > MAX_FAILED_STUDY_FRACTION:
        raise RuntimeError(f"{len(failed_studies)}/{n} studies had decode failures (> {MAX_FAILED_STUDY_FRACTION:.0%}); "
                           f"first: {failures[:3]}")
    n_masked = sum(1 for v in chosen.values() if v is None)
    n_fail = sum(f["stage"] == "decode" for f in failures)
    log(f"[{preproc}] decoded slots for {n} studies; {n_fail} series failures in {len(failed_studies)} studies; "
        f"{n_masked} slot(s) masked; {len(failures) - n_fail} series with a dropped end slice")
    return idx, failures


def load_models(model_dir, device="cuda"):
    """model_dir contains one or more <name>.pt checkpoints saved by train.py (key 'model' + 'args')."""
    models = []
    for f in sorted(os.listdir(model_dir)):
        if not f.endswith(".pt"):
            continue
        ck = torch.load(os.path.join(model_dir, f), map_location="cpu", weights_only=False)
        a = ck["args"]
        m = KneeMIL(a["backbone"], pretrained=False, n_slots=6, agg=a["agg"], use_pos=not a.get("no_pos", False))
        m.load_state_dict(ck["model"])
        m.eval().to(device)
        models.append((f, m, a))
    if not models:
        raise RuntimeError(f"no .pt checkpoints in {model_dir}")
    return models


def amp_dtype(device="cuda"):
    """bf16 (the training precision) on Ampere or newer; fp16 on older GPUs such as Kaggle's T4, where
    bf16 is only emulated (slow). Measured on 120 val studies (check_amp_precision.py): max |p - p_fp32|
    = 0.0014 for fp16 and 0.013 for bf16; per-class Spearman vs fp32 >= 0.9998 for both."""
    if str(device).startswith("cuda") and torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
        return torch.bfloat16
    return torch.float16


def window_bags(a, tta_shift=False):
    """Window-centre settings (K, lo, hi) per model: the training setting, plus (tta_shift) a second
    bag shifted by half the spacing between window centres; the bags' probabilities are averaged."""
    bags = [(a["K"], a["lo"], a["hi"])]
    if tta_shift:
        half = (a["hi"] - a["lo"]) / (2 * (a["K"] - 1))
        bags.append((a["K"], a["lo"] + half, min(1.0, a["hi"] + half)))
    return bags


@torch.no_grad()
def predict_studies(index_df, cache_dir, models, batch_size=4, workers=2, device="cuda", tta_shift=False):
    """-> list with one (N, 12) probability array per model (rows aligned with index_df)."""
    preds = []
    for name, m, a in models:
        slots = [int(s) for s in str(a["slots"]).split(",")]
        pb = []
        for K, lo, hi in window_bags(a, tta_shift):
            ds = KneeStudyDataset(index_df, cache_dir, slots, K, train=False, lo=lo, hi=hi,
                                  orient=not a.get("no_orient", False),
                              sag_pos_fix=a.get("sag_pos_fix", False))
            dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, num_workers=workers)
            p = np.zeros((len(ds), len(TARGETS)), np.float32)
            for b in dl:
                x = b["x"].to(device).float().div_(255)
                with torch.autocast("cuda", dtype=amp_dtype(device)):
                    lg = m(x, b["mask"].to(device), b["pos"].to(device))
                p[b["i"].numpy()] = torch.sigmoid(lg.float()).cpu().numpy()
            pb.append(p)
        preds.append(np.mean(pb, 0))
    return preds


def write_submission(index_df, probs, sample_sub_csv, test_csv, out_path):
    sample = pd.read_csv(sample_sub_csv)
    test = pd.read_csv(test_csv)
    sub = pd.DataFrame(probs, columns=TARGETS)
    sub.insert(0, "StudyInstanceUID", index_df.StudyInstanceUID.values)
    sub = sub.set_index("StudyInstanceUID").reindex(test.StudyInstanceUID).reset_index()
    sub = sub[list(sample.columns)]
    vals = sub[TARGETS].values
    assert list(sub.columns) == list(sample.columns), "column mismatch with sample_submission"
    assert len(sub) == len(test) and sub.StudyInstanceUID.is_unique, "row count / uniqueness mismatch"
    assert set(sub.StudyInstanceUID) == set(test.StudyInstanceUID), "study id mismatch"
    assert np.isfinite(vals).all() and (vals >= 0).all() and (vals <= 1).all(), "non-finite or out-of-range predictions"
    sub.to_csv(out_path, index=False)
    return sub


def run(cfg, log=print):
    t0 = time.time()
    assert_decoders()
    idx, f1, geos = build_test_index(cfg["test_series_csv"], cfg["test_dicom_root"], cfg["workers"], log)
    models = load_models(cfg["model_dir"], cfg["device"])
    groups = {}
    for mm in models:
        groups.setdefault(mm[2].get("preproc", "v2"), []).append(mm)
    f2, ready = [], {}
    for pp, ms in sorted(groups.items()):
        slots = sorted({int(s) for _, _, a in ms for s in str(a["slots"]).split(",")})
        idx_pp, fails = preprocess_selected(idx, geos, cfg["test_dicom_root"], os.path.join(cfg["cache_dir"], pp), slots,
                                            cfg["workers"], preproc=pp, log=log)
        f2 += [{**f, "preproc": pp} for f in fails]
        ready[pp] = idx_pp                    # same row order as idx
    t1 = time.time()
    preds = []
    for pp, ms in sorted(groups.items()):
        preds += predict_studies(ready[pp], os.path.join(cfg["cache_dir"], pp), ms, cfg["batch_size"],
                                 cfg.get("loader_workers", 2), cfg["device"], cfg.get("tta_shift", False))
    probs = np.mean(preds, 0)                 # equal weight per model
    sub = write_submission(idx, probs, cfg["sample_submission"], cfg["test_csv"], cfg["out_path"])
    t2 = time.time()
    report = {"n_studies": len(sub), "n_models": len(models), "models": [n for n, _, _ in models],
              "preproc_versions": {pp: len(ms) for pp, ms in groups.items()}, "tta_shift": bool(cfg.get("tta_shift", False)),
              "header_failures": f1, "decode_failures": f2,
              "seconds_preprocess": round(t1 - t0, 1), "seconds_predict": round(t2 - t1, 1),
              "side_source_counts": idx.side_source.value_counts().to_dict(),
              "side_conflicts": int(idx.side_conflict.sum()), "side_ambiguous": int(idx.side_ambiguous.sum())}
    json.dump(report, open(os.path.join(os.path.dirname(cfg["out_path"]) or ".", "inference_report.json"), "w"), indent=1, default=str)
    log(json.dumps({k: v for k, v in report.items() if "failures" not in k}))
    return sub, report

## 4. Run: decode self-test, index, selected-series cache, ensemble prediction, validated submission

In [ ]:
t_all = time.time()
sub, report = run(CFG)
print('total minutes', round((time.time() - t_all) / 60, 1))
sub.head()

## 5. Final checks

In [ ]:
s = pd.read_csv(CFG["out_path"])
smp = pd.read_csv(CFG["sample_submission"])
tst = pd.read_csv(CFG["test_csv"])
assert list(s.columns) == list(smp.columns)
assert len(s) == len(tst) and set(s.StudyInstanceUID) == set(tst.StudyInstanceUID)
v = s.iloc[:, 1:].values
assert np.isfinite(v).all() and (v >= 0).all() and (v <= 1).all()
print("submission OK:", s.shape, "| decode failures:", len(report["decode_failures"]), "| header failures:", len(report["header_failures"]))